# 국가별 무역구조 기반 클러스터링 분석 
### K-Means(메인) → DBSCAN(이상국 탐지) → RF/SVM(분리 일관성 확인) → 부트스트랩 ARI(구조 안정성 검증) → 군집별 심층분석

**데이터 출처**
- UN Comtrade 무역통계 (2023, 164개국, HS 챕터 1~97)
- World Bank 거시경제지표 (2023, 페이지네이션 보정으로 U~Z 국가 포함)

**최종 분석 대상**: 161개국 × 18개 변수

## 1. 데이터 로드 및 전처리

UN Comtrade CSV는 헤더(47개 필드)와 데이터 행(48개 필드) 간 불일치가 있어 `usecols=range(47)`로 보정.
World Bank 데이터는 API `per_page=500` 설정으로 U~Z 국가 누락 문제를 해결한 보정 파일 사용.

In [ ]:
import kfont
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import (silhouette_score, silhouette_samples, davies_bouldin_score,
                             accuracy_score, confusion_matrix, adjusted_rand_score)
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.neighbors import NearestNeighbors
import warnings; warnings.filterwarnings('ignore')

plt.rcParams['font.family'] = 'Nanum Gothic'
plt.rcParams['axes.unicode_minus'] = False


TRADE_PATH = 'TradeData_6_1_2026_13_1_27.csv'
WB_PATH = 'world_bank_data_2023_complete.xlsx'

trade = pd.read_csv(TRADE_PATH, low_memory=False, encoding='latin1', usecols=range(47))
trade['cmdCode'] = trade['cmdCode'].astype(int)
trade = trade[trade['cmdCode'] <= 97]
print(f'무역데이터: {trade.reporterISO.nunique()}개국, HS 챕터 1~97')

### 1-1. 무역 변수 구성

In [ ]:
# 수출/수입 집계
agg = trade.groupby(['reporterISO','flowCode','cmdCode'])['primaryValue'].sum().reset_index()
exp = agg[agg.flowCode=='X'].pivot(index='reporterISO', columns='cmdCode', values='primaryValue').fillna(0)
imp = agg[agg.flowCode=='M'].pivot(index='reporterISO', columns='cmdCode', values='primaryValue').fillna(0)
te = exp.sum(axis=1); ti = imp.sum(axis=1)

# 수출 포트폴리오 비중 & HHI
sh = exp.div(te, axis=0).fillna(0)
hhi = (sh**2).sum(axis=1)

# HS 챕터 → 13개 경제 카테고리 통합
def hs_cat(c):
    if c<=24: return 'Agri_Food'
    elif c<=27: return 'Mineral_Fuel'
    elif c<=38: return 'Chemicals'
    elif c<=40: return 'Plastics_Rubber'
    elif c<=43: return 'Hides_Leather'
    elif c<=49: return 'Wood_Paper'
    elif c<=63: return 'Textiles_Apparel'
    elif c<=67: return 'Footwear_Misc_Light'
    elif c<=71: return 'Stone_Glass_Precious'
    elif c<=83: return 'Base_Metals'
    elif c<=92: return 'Machinery_Electronics_Transport'
    elif c<=93: return 'Arms'
    else: return 'Misc_Manufactured'

catsh = pd.DataFrame(index=sh.index)
for c in sh.columns:
    k = 'EXPCAT_'+hs_cat(c)
    catsh[k] = catsh.get(k, 0) + sh[c]

tf = pd.DataFrame({'iso3':te.index, 'total_exports':te.values, 'total_imports':ti.values,
                   'HHI_export':hhi.values}).set_index('iso3')
tf = tf.join(catsh)
print(f'무역 피처: {len(tf)}개국')

### 1-2. World Bank 데이터 병합 및 결측 처리

World Bank API 기본 `per_page=50` 설정 때문에 U~Z 국가가 누락되는 문제를 `per_page=500`으로 보정하여 재추출하였다.
자원렌트(Res_Rent_GDP)는 2023년 데이터가 미발행이므로 2019~2023 범위에서 최신 가용값을 사용하였다.

In [ ]:
wb = pd.read_excel(WB_PATH).dropna(subset=['iso3']).set_index('iso3')
m = tf.join(wb, how='inner')
m['Trade_Openness_GDP'] = (m.total_exports + m.total_imports) / m.gdp_usd * 100

# 결측 대치 (중앙값)
macro_cols = ['Mfg_VA_GDP','Res_Rent_GDP','Trade_Balance_GDP']
for col in macro_cols:
    n_miss = m[col].isna().sum()
    if n_miss > 0: print(f'  {col}: {n_miss}개국 중앙값 대치')
    m[col] = SimpleImputer(strategy='median').fit_transform(m[[col]])

m['Trade_Openness_GDP_capped'] = m['Trade_Openness_GDP'].clip(upper=m['Trade_Openness_GDP'].quantile(0.99))
feat = ['Mfg_VA_GDP','Res_Rent_GDP','Trade_Balance_GDP','HHI_export','Trade_Openness_GDP_capped'] + \
       [c for c in m.columns if c.startswith('EXPCAT_')]

X = m[feat].values; Xs = StandardScaler().fit_transform(X)
print(f'\n최종 분석: {len(m)}개국, {len(feat)}개 변수')

### 1-3. 데이터 병합 현황 검증

In [ ]:
trade_iso = set(trade['reporterISO'].dropna().unique())
wb_all = pd.read_excel(WB_PATH)
wb_valid_iso = set(wb_all.dropna(subset=['iso3','gdp_usd'])['iso3'].unique())

print(f'UN Comtrade 국가 수: {len(trade_iso)}개')
print(f'World Bank 국가 수 (GDP 유효): {len(wb_valid_iso)}개')
print(f'병합(inner join) 결과: {len(m)}개국')

dropped = trade_iso - set(m.index)
print(f'\n무역데이터에는 있으나 병합에서 탈락한 국가: {len(dropped)}개')
for iso in sorted(dropped):
    name = trade[trade.reporterISO==iso]['reporterDesc'].iloc[0] if 'reporterDesc' in trade.columns else iso
    print(f'  {iso}: {name}')

## 2. PCA 탐색: 96개 HS 챕터 수출비중에 대한 차원축소 검토

In [ ]:
pca96 = PCA().fit(StandardScaler().fit_transform(sh.values))
cumvar = np.cumsum(pca96.explained_variance_ratio_)

fig, ax = plt.subplots(figsize=(8,4))
ax.plot(range(1,len(cumvar)+1), cumvar, 'b-o', markersize=3)
ax.axhline(0.8, ls='--', c='r', alpha=0.5)
ax.set_xlabel('주성분 수'); ax.set_ylabel('누적 설명분산')
ax.set_title('96개 HS 챕터 수출비중에 대한 PCA 누적설명분산')
n80 = int(np.argmax(cumvar >= 0.8) + 1)
ax.annotate(f'{n80}개 PC → 80%', (n80, 0.8), fontsize=9, color='red')
plt.tight_layout(); plt.show()
print(f'20개 주성분 → {cumvar[19]:.1%} 설명')

**검토 결과**: 96개 HS 챕터 비중 변수는 분산이 매우 분산되어 있어 PCA로 명확한 저차원 구조를 얻기 어렵다.
이에 13개 경제 카테고리 통합 변수를 사용한다.

### 2-1. 변수 간 상관관계 점검

In [ ]:
corr = m[feat].corr()
fig, ax = plt.subplots(figsize=(10,8))
mask_tri = np.triu(np.ones_like(corr, dtype=bool), k=1)
labels = [f.replace('EXPCAT_','').replace('_',' ')[:16] for f in feat]
sns.heatmap(corr, mask=mask_tri, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            xticklabels=labels, yticklabels=labels, ax=ax, annot_kws={'size':7})
ax.set_title('최종 18개 클러스터링 변수 상관행렬')
plt.tight_layout(); plt.show()
print(f'Mfg_VA ↔ Res_Rent: {corr.loc["Mfg_VA_GDP","Res_Rent_GDP"]:.3f}')
print(f'Mineral_Fuel ↔ Res_Rent: {corr.loc["EXPCAT_Mineral_Fuel","Res_Rent_GDP"]:.3f}')

**발견**: 제조업 부가가치 비중과 자원렌트 비중의 상관계수는 -0.12로 약하다. 이는 일부 국가가 제조업과 자원수출을 동시에 영위하는 복합형 경제구조를 가지고 있음을 시사한다.

### 2-5. 무역개방도 분포 및 이상치 절단

무역개방도((수출+수입)/GDP)는 우편향 분포를 보이며, 중계무역·환적 허브 국가가 극단값을 형성한다.
표준화 전 분포를 시각적으로 확인하고, 99th percentile 절단의 효과를 검증한다.

In [ ]:
raw_openness = m['Trade_Openness_GDP']
cap_value = raw_openness.quantile(0.99)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].hist(raw_openness, bins=40, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].axvline(cap_value, ls='--', c='red', lw=1.5, label=f'99th pct cap = {cap_value:.0f}%')
axes[0].axvline(raw_openness.mean(), ls=':', c='green', lw=1.5, label=f'평균 = {raw_openness.mean():.0f}%')
axes[0].set_xlabel('무역개방도 (%)'); axes[0].set_ylabel('국가 수')
axes[0].set_title('무역개방도 분포 (절단 전)')
axes[0].legend(fontsize=9)

bp = axes[1].boxplot([raw_openness, raw_openness.clip(upper=cap_value)],
                     labels=['절단 전', '절단 후 (99%)'], patch_artist=True, widths=0.5)
for patch, color in zip(bp['boxes'], ['lightcoral', 'lightgreen']): patch.set_facecolor(color)
axes[1].set_ylabel('무역개방도 (%)'); axes[1].set_title('99% 절단 효과')

plt.tight_layout(); plt.show()

print(f'평균: {raw_openness.mean():.1f}%, 표준편차: {raw_openness.std():.1f}%, 최댓값: {raw_openness.max():.1f}%')
print(f'99th percentile: {cap_value:.1f}%')
capped_countries = raw_openness[raw_openness > cap_value]
print(f'\n절단 대상국 ({len(capped_countries)}개):')
for iso, val in capped_countries.items():
    print(f'  {iso} ({m.loc[iso,"country_name"]}): {val:.1f}% -> {cap_value:.1f}%')

## 3. K-Means 클러스터링 (메인)

### 3-1. 최적 군집 수(k) 탐색

In [ ]:
K = range(2,11)
inertia, sils, dbs = [], [], []
for k in K:
    km_ = KMeans(k, n_init=50, random_state=42).fit(Xs)
    inertia.append(km_.inertia_)
    sils.append(silhouette_score(Xs, km_.labels_))
    dbs.append(davies_bouldin_score(Xs, km_.labels_))

fig, axes = plt.subplots(1,3,figsize=(14,4))
axes[0].plot(K, inertia, 'bo-'); axes[0].set_title('Elbow (관성)'); axes[0].set_xlabel('k')
axes[1].plot(K, sils, 'go-'); axes[1].set_title('Silhouette Score'); axes[1].set_xlabel('k')
axes[2].plot(K, dbs, 'ro-'); axes[2].set_title('Davies-Bouldin Index'); axes[2].set_xlabel('k')
for ax in axes: ax.axvline(5, ls='--', alpha=0.3, c='gray')
plt.tight_layout(); plt.show()

### 3-2. K-Means(k=5) 최종 결과

In [ ]:
km = KMeans(5, n_init=50, random_state=42)
lab = km.fit_predict(Xs)
m['cluster'] = lab
sil_avg = silhouette_score(Xs, lab)
print(f'평균 실루엣 점수: {sil_avg:.3f}')
print(f'군집 크기: {list(np.bincount(lab))}')

fig, ax = plt.subplots(figsize=(8,6))
sil_vals = silhouette_samples(Xs, lab)
y_lower = 10; colors = plt.cm.Set2(np.linspace(0,1,5))
for i in range(5):
    ith = np.sort(sil_vals[lab==i])
    y_upper = y_lower + len(ith)
    ax.fill_betweenx(np.arange(y_lower,y_upper), 0, ith, alpha=0.7, color=colors[i])
    ax.text(-0.05, y_lower+0.5*len(ith), f'C{i} (n={len(ith)})', fontsize=9)
    y_lower = y_upper + 10
ax.axvline(sil_avg, ls='--', c='red')
ax.set_xlabel('실루엣 계수'); ax.set_title(f'K-Means(k=5) 실루엣 플롯 (평균={sil_avg:.3f})')
plt.tight_layout(); plt.show()

### 3-3. 군집 중심값 해석

In [ ]:
scaler = StandardScaler().fit(X)
centers = pd.DataFrame(scaler.inverse_transform(km.cluster_centers_), columns=feat)

lm = {}
for i in range(5):
    c = centers.loc[i]
    if c['EXPCAT_Mineral_Fuel'] > 0.5: lm[i] = '자원수출 의존형'
    elif c['EXPCAT_Machinery_Electronics_Transport'] > 0.3: lm[i] = '제조업/가공무역형'
    elif c['EXPCAT_Textiles_Apparel'] > 0.2: lm[i] = '섬유의류 특화형'
    elif c['EXPCAT_Stone_Glass_Precious'] > 0.3: lm[i] = '귀금속/원자재 집중형'
    elif c['EXPCAT_Agri_Food'] > 0.3: lm[i] = '농식품 수출형'
    else: lm[i] = f'기타{i}'

for i in range(5):
    c = centers.loc[i]; members = m[m.cluster==i]
    top = members.sort_values('total_exports', ascending=False).head(6)
    names = ', '.join([f"{idx}({m.loc[idx,'country_name']})" for idx in top.index])
    print(f'\n군집{i} ({lm[i]}, n={len(members)}):')
    print(f'  Mfg={c.Mfg_VA_GDP:.1f}% | Res={c.Res_Rent_GDP:.1f}% | HHI={c.HHI_export:.2f} | 개방도={c.Trade_Openness_GDP_capped:.0f}%')
    print(f'  농식품={c.EXPCAT_Agri_Food:.1%} | 연료={c.EXPCAT_Mineral_Fuel:.1%} | 기계={c.EXPCAT_Machinery_Electronics_Transport:.1%}')
    print(f'  대표: {names}')

### 3-4. PCA 2D 투영 시각화

In [ ]:
pca2d = PCA(2); pc = pca2d.fit_transform(Xs)
m['pc1']=pc[:,0]; m['pc2']=pc[:,1]
ve = pca2d.explained_variance_ratio_

fig, ax = plt.subplots(figsize=(10,7))
for i in range(5):
    mask = m.cluster==i
    ax.scatter(m.loc[mask,'pc1'], m.loc[mask,'pc2'],
              label=f'C{i}: {lm[i]} ({mask.sum()})', alpha=0.7, s=40)
for iso in ['KOR','CHN','DEU','USA','GBR','JPN','SAU','NGA','BRA','IND','VNM','ARE','PAK']:
    if iso in m.index:
        ax.annotate(iso, (m.loc[iso,'pc1'], m.loc[iso,'pc2']), fontsize=7, alpha=0.8)
ax.set_xlabel(f'PC1 ({ve[0]:.1%})'); ax.set_ylabel(f'PC2 ({ve[1]:.1%})')
ax.set_title(f'K-Means 군집 PCA 2D 투영 (누적 {sum(ve):.1%})')
ax.legend(fontsize=8); plt.tight_layout(); plt.show()

### 3-5. 군집별 수출 품목 구성

In [ ]:
cc = [c for c in m.columns if c.startswith('EXPCAT_')]
cm = m.groupby('cluster')[cc].mean()
cm.columns = [c.replace('EXPCAT_','') for c in cm.columns]
fig, ax = plt.subplots(figsize=(10,5))
cm.T.plot(kind='barh', stacked=True, ax=ax, colormap='Set2')
ax.set_xlabel('평균 수출비중'); ax.set_title('군집별 평균 수출 카테고리 비중')
ax.legend([f'C{i}: {lm[i]}' for i in range(5)], loc='lower right', fontsize=7)
plt.tight_layout(); plt.show()

## 4. DBSCAN: 밀도 기반 이상국 탐지

### 4-1. eps 파라미터 선정

In [ ]:
nn = NearestNeighbors(n_neighbors=5).fit(Xs)
dists, _ = nn.kneighbors(Xs); k_dists = np.sort(dists[:,-1])

fig, ax = plt.subplots(figsize=(8,4))
ax.plot(k_dists, 'b-'); ax.axhline(4.2, ls='--', c='r', alpha=0.5, label='eps=4.2')
ax.set_xlabel('국가 (오름차순)'); ax.set_ylabel('5-거리')
ax.set_title('DBSCAN eps 선정을 위한 5-거리 그래프'); ax.legend()
plt.tight_layout(); plt.show()

### 4-2. 민감도 분석 및 이상국 식별

In [ ]:
print('=== DBSCAN 민감도 분석 ===')
for eps in [3.6, 3.8, 4.0, 4.2, 4.4, 4.6]:
    for ms in [4, 5, 6]:
        dl = DBSCAN(eps=eps, min_samples=ms).fit_predict(Xs)
        nc = len(set(dl))-(1 if -1 in dl else 0); no = (dl==-1).sum()
        print(f'  eps={eps} ms={ms}: 군집={nc}, 이상국={no}')

db = DBSCAN(eps=4.2, min_samples=5).fit_predict(Xs)
m['dbscan_outlier'] = (db == -1)
outliers = m[m.dbscan_outlier]
print(f'\n이상국 ({len(outliers)}개):')
for idx, row in outliers.iterrows():
    print(f'  {idx}({row.get("country_name",idx)}) → 군집{int(row.cluster)}')

# PCA 시각화
fig, ax = plt.subplots(figsize=(10,7))
normal = m[~m.dbscan_outlier]
ax.scatter(normal.pc1, normal.pc2, c='lightblue', alpha=0.5, s=30, label='정상')
ax.scatter(outliers.pc1, outliers.pc2, c='red', marker='x', s=80, label=f'이상국 ({len(outliers)}개)')
for idx in outliers.index:
    ax.annotate(idx, (m.loc[idx,'pc1'], m.loc[idx,'pc2']), fontsize=7, color='red')
ax.set_title('PCA 평면상의 DBSCAN 이상국'); ax.legend()
plt.tight_layout(); plt.show()

## 5. RF/SVM: 군집 분리 일관성 확인

K-Means 군집 레이블을 RF/SVM으로 재현할 수 있는지 확인한다. 동일 피처 공간에서의 복원이므로 '군집이 실재하는 구조'라는 강한 주장보다는,
**군집 경계가 일관적이고 분리 가능하다**는 것을 확인하는 용도로 해석한다. 경계에 위치한 국가 식별에 활용한다.

In [ ]:
skf = StratifiedKFold(5, shuffle=True, random_state=42)
rf_pred = cross_val_predict(RandomForestClassifier(300, class_weight='balanced', random_state=42), Xs, lab, cv=skf)
svm_pred = cross_val_predict(SVC(kernel='rbf', C=1.0, class_weight='balanced'), Xs, lab, cv=skf)
rf_acc = accuracy_score(lab, rf_pred); svm_acc = accuracy_score(lab, svm_pred)
print(f'RF 정확도: {rf_acc:.1%}')
print(f'SVM 정확도: {svm_acc:.1%}')

# 혼동행렬
fig, axes = plt.subplots(1,2,figsize=(12,5))
for ax, pred, name, acc in [(axes[0],rf_pred,'Random Forest',rf_acc),(axes[1],svm_pred,'SVM (RBF)',svm_acc)]:
    sns.heatmap(confusion_matrix(lab, pred), annot=True, fmt='d', cmap='Blues', ax=ax)
    ax.set_xlabel('예측'); ax.set_ylabel('실제'); ax.set_title(f'{name} (정확도: {acc:.1%})')
plt.tight_layout(); plt.show()

# 변수 중요도
rf_full = RandomForestClassifier(300, class_weight='balanced', random_state=42).fit(Xs, lab)
imp_s = pd.Series(rf_full.feature_importances_, index=feat).sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(8,6))
imp_s.tail(12).plot(kind='barh', ax=ax)
ax.set_title('Random Forest 변수 중요도 (Top 12)'); ax.set_xlabel('중요도')
plt.tight_layout(); plt.show()

# 경계국 식별
m['boundary'] = (rf_pred != lab) | (svm_pred != lab)
print(f'\n경계국: {m.boundary.sum()}개 ({m.boundary.sum()/len(m):.1%})')

## 6. 부트스트랩 ARI: 군집 구조 안정성의 독립 검증

RF/SVM 검증은 동일 피처 공간 내 복원이라는 한계가 있다. 이를 보완하기 위해 **부트스트랩 서브샘플링**을 통한 독립적 안정성 검증을 수행한다.
80% 랜덤 서브샘플에서 K-Means(k=5)를 100회 반복 실행하고, 원래 레이블과의 Adjusted Rand Index(ARI)로 구조 안정성을 측정한다.

ARI 해석 기준: 0.9+ 매우 안정, 0.7~0.9 양호, 0.5~0.7 보통, <0.5 불안정

In [ ]:
n_boot = 100
ari_scores = []
np.random.seed(42)
for b in range(n_boot):
    idx = np.random.choice(len(Xs), size=int(len(Xs)*0.8), replace=False)
    km_sub = KMeans(5, n_init=20, random_state=b).fit(Xs[idx])
    ari_scores.append(adjusted_rand_score(lab[idx], km_sub.labels_))
ari_scores = np.array(ari_scores)

fig, ax = plt.subplots(figsize=(8,4))
ax.hist(ari_scores, bins=20, edgecolor='black', alpha=0.7)
ax.axvline(ari_scores.mean(), c='red', ls='--', label=f'평균 ARI={ari_scores.mean():.3f}')
ax.set_xlabel('Adjusted Rand Index'); ax.set_ylabel('빈도')
ax.set_title(f'부트스트랩 ARI 분포 (100회, 80% 서브샘플)')
ax.legend(); plt.tight_layout(); plt.show()

print(f'ARI: {ari_scores.mean():.3f} ± {ari_scores.std():.3f}')
print(f'최소: {ari_scores.min():.3f}, 최대: {ari_scores.max():.3f}')

## 7. 군집별 심층 분석

In [ ]:
# 군집별 거시지표 분포
fig, axes = plt.subplots(2,2,figsize=(12,8))
for ax, col, title in zip(axes.flat,
    ['Mfg_VA_GDP','Res_Rent_GDP','HHI_export','Trade_Openness_GDP_capped'],
    ['제조업 VA/GDP (%)','자원렌트/GDP (%)','수출 HHI','무역개방도 (%)']):
    data = [m.loc[m.cluster==i, col].values for i in range(5)]
    bp = ax.boxplot(data, labels=[f'C{i}' for i in range(5)], patch_artist=True)
    for patch, color in zip(bp['boxes'], colors): patch.set_facecolor(color)
    ax.set_title(title)
plt.suptitle('군집별 핵심 거시지표 분포', fontsize=13)
plt.tight_layout(); plt.show()

### 7-1. 군집 안정성 종합

In [ ]:
print(f'{"군집":<25} {"n":>4} {"DBSCAN이상":>12} {"경계국":>12} {"안정성":>8}')
print('-'*65)
for i in range(5):
    mask = m.cluster==i; n = mask.sum()
    db_n = int(m.loc[mask,'dbscan_outlier'].sum()); bd_n = int(m.loc[mask,'boundary'].sum())
    stab = '매우높음' if db_n==0 and bd_n<=2 else '높음' if bd_n/n<0.1 else '중간' if bd_n/n<0.25 else '낮음'
    print(f'C{i} {lm[i]:<20} {n:>4} {db_n:>4}({db_n/n:>5.0%}) {bd_n:>4}({bd_n/n:>5.0%}) {stab:>8}')

### 7-2. 특정 경계 국가 중심의 벤치마킹 갭(Gap) 분석

5장에서 식별된 29개 경계국 중, C3(제조업/가공무역형)에 속한 국가를 대상으로 군집 중심(centroid)과의 변수별 격차를 분해한다.
**주의**: 이는 인과적 처방이 아니라 현재 시점의 distance decomposition이다.

In [ ]:
# C3 경계국 확인
c3_id = m.loc['JOR', 'cluster']
c3_boundary = m[(m.cluster == c3_id) & (m.boundary)]
print(f'C3({lm[c3_id]}) 경계국 ({len(c3_boundary)}개): {list(c3_boundary.index)}')

# 대상국: 요르단 (중동 신흥 산업화 개도국, 경계국, 중심거리 주변부)
target = 'JOR'
print(f'\n{target} 군집: C{c3_id} ({lm[c3_id]})')
print(f'{target} 경계국 여부: {m.loc[target, "boundary"]}')

# 군집 중심까지 거리 순위
c3_members = m[m.cluster == c3_id].copy()
center = km.cluster_centers_[c3_id]
idx_map = {iso: i for i, iso in enumerate(m.index)}
dists = np.linalg.norm(Xs[[idx_map[i] for i in c3_members.index]] - center, axis=1)
c3_members['dist_to_center'] = dists
c3_members = c3_members.sort_values('dist_to_center')
rank = list(c3_members.index).index(target) + 1
print(f'{target} 중심거리 순위: {rank}/{len(c3_members)} (1=중심에 가장 가까움)')
print(f'중심에 가장 가까운 5개국: {list(c3_members.index[:5])}')

In [ ]:
# 변수별 갭(Gap) 분해: 요르단 vs C3 중심값
gap_cols = ['Trade_Openness_GDP_capped','Mfg_VA_GDP','HHI_export',
            'EXPCAT_Machinery_Electronics_Transport','EXPCAT_Chemicals',
            'EXPCAT_Textiles_Apparel','EXPCAT_Stone_Glass_Precious']
c3_centroid_real = centers.loc[c3_id]

gap_table = []
for col in gap_cols:
    jor_val = m.loc[target, col]
    c3_val = c3_centroid_real[col]
    gap_table.append({'변수': col, target: jor_val, 'C3_중심': c3_val, 'Gap': jor_val - c3_val})

gap_df = pd.DataFrame(gap_table).set_index('변수')
print(gap_df.round(3))

# 시각화: 갭 막대그래프
fig, ax = plt.subplots(figsize=(9,5))
colors_gap = ['red' if g < 0 else 'green' for g in gap_df['Gap']]
ax.barh(gap_df.index, gap_df['Gap'], color=colors_gap, alpha=0.7)
ax.axvline(0, color='black', lw=0.8)
ax.set_xlabel('Gap (요르단 - C3 중심)')
ax.set_title(f'{target} vs C3({lm[c3_id]}) 군집 중심 변수별 격차')
plt.tight_layout(); plt.show()

**해석**: 기계/전자/운송장비 수출비중에서 가장 큰 음(-)의 격차(-32.1%p)를 보이며, 화학(+20.6%p)·섬유의류(+12.3%p) 비중은 군집 평균을 상회한다. 
이는 요르단이 C3 군집에 속하면서도 화학·1차가공품 중심의 중간 단계에 머물러 있음을 시사한다. 
단, 이 갭 수치를 '몇 %p 조정하면 군집 중심으로 이동한다'는 인과적 처방으로 해석해서는 안 되며, 
현재 시점의 기술적 거리 분해 결과로 한정하여 이해해야 한다.

### 7-3. 군집별 GDP 성장률 비교

제안서의 핵심 질문인 '어떤 경제 체질이 성장에 유리한가'를 검증하기 위해 군집별 GDP 성장률을 비교한다.
**주의**: 평균은 극단값(마카오 75% 등 기저효과)에 취약하므로 중앙값을 기준으로 해석한다.

In [ ]:
from scipy import stats

growth_stats = m.groupby('cluster')['GDP_Growth'].agg(['mean','median','std','count'])
growth_stats.index = [lm[i] for i in range(5)]
print(growth_stats.round(2))

c0_sorted = m[m.cluster==0][['country_name','GDP_Growth']].sort_values('GDP_Growth', ascending=False)
print('\nC0(섬유의류) 성장률 상위 - 평균 왜곡 원인 확인:')
print(c0_sorted.head(3))

# Kruskal-Wallis
groups = [m.loc[m.cluster==i, 'GDP_Growth'].dropna().values for i in range(5)]
h_stat, p_kw = stats.kruskal(*groups)
print(f'\nKruskal-Wallis 검정: H={h_stat:.3f}, p={p_kw:.4f}')

fig, ax = plt.subplots(figsize=(8,5))
data = [m.loc[m.cluster==i,'GDP_Growth'].values for i in range(5)]
bp = ax.boxplot(data, labels=[lm[i] for i in range(5)], patch_artist=True, showfliers=True)
for patch, color in zip(bp['boxes'], colors): patch.set_facecolor(color)
ax.set_ylabel('GDP 성장률 (%)'); ax.set_title('군집별 GDP 성장률 분포 (마카오 등 극단값 포함)')
ax.axhline(0, color='gray', ls=':', alpha=0.5)
plt.tight_layout(); plt.show()

**해석**: 평균 기준으로는 C0(섬유의류, 9.9%)이 가장 높아 보이지만 이는 마카오(75.3%, 코로나 회복 기저효과) 단일 국가에 의한 왜곡이다. 
마카오를 포함한 중앙값으로 비교하면 제조업/가공무역형(1.82%)만 유의하게 낮고, 나머지 4개 군집은 3.5~3.9%로 비슷한 수준이다. 
이는 이미 산업화·가공무역 구조가 성숙한 경제(중국, 독일, 일본 등 선진 제조국 다수 포함)일수록 성장 여력이 상대적으로 낮은, 
전형적인 '성장 둔화(growth deceleration)' 패턴으로 해석할 수 있다.

### 7-4. 변수 중요도 가설 대조: 거시구조 변수 vs 수출 카테고리 변수

제안서는 거시구조 변수의 중요도 순서를 'Mfg_VA_GDP > Res_Rent_GDP > Trade_Balance_GDP > HHI_export'로 예상하였다. 
실제 RF 변수 중요도와 대조한다.

In [ ]:
rf_full2 = RandomForestClassifier(300, class_weight='balanced', random_state=42).fit(Xs, lab)
imp_full = pd.Series(rf_full2.feature_importances_, index=feat).sort_values(ascending=False)

macro_vars = ['Mfg_VA_GDP','Res_Rent_GDP','Trade_Balance_GDP','HHI_export']
print('=== 거시구조 변수: 예상 순위 vs 실제 순위 (전체 18개 중) ===')
expected_rank = {'Mfg_VA_GDP':1,'Res_Rent_GDP':2,'Trade_Balance_GDP':3,'HHI_export':4}
for v in macro_vars:
    actual_rank = list(imp_full.index).index(v)+1
    print(f'  {v}: 예상 {expected_rank[v]}위 -> 실제 {actual_rank}/18위 (중요도 {imp_full[v]:.4f})')

print('\n=== 실제 상위 5개 변수 ===')
print(imp_full.head(5))

**해석**: 가설과 정반대로, 거시구조 변수(Mfg_VA_GDP, Trade_Balance_GDP 등)는 중요도 하위권(11~15위)에 머물렀고, 
수출 카테고리 비중 변수(귀금속/보석, 광물연료, 섬유/의류)가 상위 1~3위를 차지하였다. 
이는 18개 변수 중 13개가 수출 카테고리 변수이고 K-Means 군집 자체가 수출 구조 차이로 가장 뚜렷하게 갈리기 때문으로, 
거시지표보다 '무엇을 수출하는가'가 군집 구분에 훨씬 직접적인 정보를 제공함을 보여준다. 
단, Res_Rent_GDP는 18개 중 4위로 거시구조 변수 중 유일하게 상위권에 위치하여, 자원 의존도만큼은 가설처럼 중요한 동인으로 확인되었다.

### 7-5. 군집 간 보완적 무역구조 분석 (Trade Complementarity Index)

제안서의 '상호 보완적 무역 파트너 발굴' 목표를 검증한다. 표준 무역보완지수(TCI, Trade Complementarity Index)를 사용한다.

TCI(j,k) = 100 × (1 − 0.5 × Σ|수출비중_j − 수입비중_k|), 0~100 범위, 높을수록 j의 수출구조가 k의 수입구조와 잘 맞음.

**주의**: 군집 평균으로 계산하면 군집간 수입구조가 매우 동질적이어서(아래 검증) 분석이 무의미해지므로, 국가 단위로 계산한 뒤 군집간 평균을 취한다.

In [ ]:
imp_piv = agg[agg.flowCode=='M'].pivot(index='reporterISO', columns='cmdCode', values='primaryValue').fillna(0)
ti_ = imp_piv.sum(axis=1)
imp_sh = imp_piv.div(ti_, axis=0).fillna(0)
impcat = pd.DataFrame(index=imp_sh.index)
for c in imp_sh.columns:
    k = 'IMPCAT_'+hs_cat(c); impcat[k] = impcat.get(k,0) + imp_sh[c]
impcat = impcat.loc[impcat.index.isin(m.index)]
m2 = m.join(impcat)
imp_cols = [c for c in impcat.columns if c.startswith('IMPCAT_')]

imp_profile_check = m2.groupby('cluster')[imp_cols].mean()
exp_profile_check = m2.groupby('cluster')[exp_cols].mean()
print('수입 프로필의 군집간 표준편차 (최대):', imp_profile_check.std().max().round(4))
print('수출 프로필의 군집간 표준편차 (최대):', exp_profile_check.std().max().round(4))
print('-> 수입구조가 수출구조보다 훨씬 동질적 -> 군집평균 기반 보완분석은 부적절, 국가단위로 계산')

In [ ]:
# 국가 단위 TCI 계산
X_exp = m2[exp_cols].values.copy()
X_imp = m2[imp_cols].values.copy()
n_c = len(m2)
TCI = np.zeros((n_c, n_c))
for j in range(n_c):
    diff = np.abs(X_exp[j][None,:] - X_imp)
    TCI[j,:] = 100*(1 - 0.5*diff.sum(axis=1))

idx_list = list(m2.index)
cluster_of = m2['cluster'].to_dict()

# 군집간 평균 TCI (국가쌍 단위 계산 후 평균)
cross_tci = pd.DataFrame(index=range(5), columns=range(5), dtype=float)
for ci in range(5):
    for cj in range(5):
        ii = [n for n in idx_list if cluster_of[n]==ci]
        jj = [n for n in idx_list if cluster_of[n]==cj]
        sub = pd.DataFrame(TCI, index=idx_list, columns=idx_list).loc[ii, jj].values
        if ci == cj:
            sub = sub[~np.eye(len(ii), dtype=bool)] if len(ii)>1 else sub
        cross_tci.loc[ci,cj] = sub.mean()
cross_tci.index = [lm[i] for i in range(5)]
cross_tci.columns = [lm[i] for i in range(5)]
print('=== 군집간 평균 TCI (수출 군집(행) -> 수입 군집(열)) ===')
print(cross_tci.round(1))

print('\n=== 자기 군집을 제외한 최고 평균 보완 상대 ===')
for ci in range(5):
    row = cross_tci.iloc[ci].copy()
    row[lm[ci]] = -1
    best = row.idxmax()
    print(f'{lm[ci]} 수출 -> {best} 수입 (평균 TCI={row[best]:.1f})')

**해석**: 가장 두드러진 보완관계는 제조업/가공무역형의 수출이 자원수출 의존형의 수입과 맺는 관계(평균 TCI=65.7)이다. 
그러나 이는 비대칭적이다 — 반대로 자원수출 의존형의 수출이 제조업/가공무역형의 수입과 맺는 TCI는 34.9에 불과하다. 
즉 제조업 국가의 다변화된 수출(기계·화학 등)은 자원수출국이 필요로 하는 수입품과 잘 맞지만, 
자원수출국의 좁은 수출 포트폴리오(연료·광물 중심)는 제조업국이 수입하는 다양한 품목과는 상대적으로 덜 맞는다. 
이는 자원-제조업 양 진영 간 무역관계가 일방적 보완 구조에 가깝다는 것을 시사하며, 실제 사우디·UAE 등 자원수출국이 
독일·한국 등 제조업국으로부터 기계·전자제품을 대량 수입하는 현실과 부합한다.

**주의**: 개별 국가쌍 단위로 보면 네덜란드(로테르담 환적항)처럼 중계무역 비중이 큰 국가가 다수 국가와 높은 TCI를 보이는데, 
이는 실제 양자 무역관계가 아니라 환적·재수출 통계가 반영된 결과일 수 있어 해석에 주의가 필요하다.

## 8. 최종 데이터 저장

In [ ]:
m.to_csv('final_clustering_results.csv')
print(f'✓ 저장 완료: final_clustering_results.csv ({len(m)}개국)')
m[['country_name','cluster','dbscan_outlier','boundary']].head(20)

## 9. 결론 및 시사점

1. World Bank 데이터 보정(per_page=500)으로 기존 150개국 → **161개국**으로 분석 범위를 확대하였으며, 미국·영국·베트남·UAE 등 주요 교역국이 분석에 포함되었다.

2. K-Means(k=5)는 자원수출 의존형, 제조업/가공무역형, 농식품 수출형, 섬유의류 특화형, 귀금속/원자재 집중형이라는 경제학적으로 해석 가능한 5개 무역구조 유형을 도출하였다.

3. **부트스트랩 ARI(0.70±0.12)**는 서브샘플에서도 군집 구조가 양호한 수준으로 재현됨을 보여준다. 이는 RF/SVM의 동일 피처 공간 내 복원(순환논증 가능성)과 달리, 데이터 부분집합에서의 독립적 안정성 검증이다.

4. RF(85.7%)/SVM(91.9%)은 군집 경계의 분리 일관성을 확인하였으며, 동시에 29개 경계국(18.0%)을 식별하였다.

5. DBSCAN은 모든 파라미터 조합에서 군집 수 1개로 나타나, 밀도 기반 관점에서 국가 무역구조는 **하나의 연속적 스펙트럼**을 형성하며, K-Means의 5개 군집은 이 스펙트럼을 해석 가능한 단위로 분할한 것이다.

6. 귀금속/원자재 집중형은 가장 안정적인 군집이며, 섬유의류 특화형은 가장 불안정한 군집으로 나타났다.

### 한계
- 단일 연도(2023년) 횡단면 분석으로 시계열적 변화를 포착하지 못함
- HS 챕터를 13개로 통합하는 기준에 일부 분석자 판단이 개입
- k=5는 여러 지표를 종합한 판단이며 절대적 최적값은 아님